## Structured Data Cleaning

In this notebook we will apply generic transformations to our structured datasets stored in MinIO. Specifically, we will clean and normalize the data (standardizing column names, handling missing values, and normalizing text fields such as case and whitespace), while ensuring consistency across all collections before analysis. All transformations will be performed using Apache Spark for distributed processing, with the final cleaned results stored back into MongoDB.

**Importing Useful Libraries**

In [ ]:
import os
import re
import ast
import boto3
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

In [ ]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)


DELTA_VERSION = "4.1.0" 
CLICKHOUSE_CONNECTOR_VERSION = "0.8.0" 


spark = SparkSession.builder \
    .appName("trusted_zone-structured") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", 
            f"org.apache.hadoop:hadoop-aws:3.3.4,"
            f"com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            f"io.delta:delta-spark_2.13:{DELTA_VERSION},"
            f"com.clickhouse.spark:clickhouse-spark-runtime-3.4_2.13:{CLICKHOUSE_CONNECTOR_VERSION},"
            f"com.clickhouse:clickhouse-jdbc:0.6.5") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.catalog.clickhouse", "com.clickhouse.spark.ClickHouseCatalog") \
    .config("spark.sql.catalog.clickhouse.host", "clickhouse") \
    .config("spark.sql.catalog.clickhouse.http_port", "8123") \
    .config("spark.sql.catalog.clickhouse.user", "analytics") \
    .config("spark.sql.catalog.clickhouse.password", "analytics_secret") \
    .config("spark.sql.catalog.clickhouse.database", "bi_analytics") \
    .config("spark.hadoop.fs.s3a.endpoint", endpoint) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.access.key", access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", secret_key) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

# Normalize Hadoop configuration values (e.g., converting "60s" to "60") to prevent version mismatch errors
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
for item in hadoop_conf.iterator():
    key = item.getKey()
    value = item.getValue()

    if isinstance(value, str) and (value.endswith("s") or value.endswith("h")):
        numeric_value = "".join(char for char in value if char.isdigit())
        hadoop_conf.set(key, numeric_value)

In [ ]:
def auto_create_ch_table_from_delta(delta_path_or_name, ch_db, ch_table_name, order_by_cols=[]):
    """
    Infers the schema from a Delta dataset directly via s3a, 
    generates ClickHouse DDL, and creates the table via Spark SQL.
    """
    print(delta_path_or_name)
    df = spark.read.format("delta").load(delta_path_or_name)
    print(df)
    type_mapping = {
        StringType: "String", IntegerType: "Int32", LongType: "Int64",
        FloatType: "Float32", DoubleType: "Float64", BooleanType: "UInt8",
        DateType: "Date", TimestampType: "DateTime", ShortType: "Int16", ByteType: "Int8"
    }
    df.head()
    ch_columns = []
    for field in df.schema.fields:
        field_name = field.name
        field_type = type(field.dataType)
        
        if field_type == ArrayType:
            element_type = type(field.dataType.elementType)
            ch_sub_type = type_mapping.get(element_type, "String")
            ch_type = f"Array({ch_sub_type})"
        else:
            ch_type = type_mapping.get(field_type, "String")
            
        ch_columns.append(f"`{field_name}` {ch_type}")
    
    columns_sql = ",\n    ".join(ch_columns)
    order_by_sql = ", ".join([f"`{c}`" for c in order_by_cols]) if order_by_cols else df.schema.fields[0].name
        
    full_ch_table_path = f"clickhouse.{ch_db}.{ch_table_name}"
    spark.sql(f"CREATE DATABASE IF NOT EXISTS clickhouse.{ch_db}")
    
    create_table_ddl = f"""
    CREATE TABLE IF NOT EXISTS {full_ch_table_path} (
        {columns_sql}
    )
    USING clickhouse
    TBLPROPERTIES (engine = 'MergeTree()', order_by = '{order_by_sql}')
    """
    spark.sql(create_table_ddl)
    print(f"[SUCCESS] ClickHouse table defined: {full_ch_table_path}")
    return df

In [ ]:
import re

# 1. Access Spark's underlying Hadoop FileSystem API to scan the target s3a path
sc = spark.sparkContext
base_path_str = "s3a://landing-zone/persistent-landing/structured/"
path_obj = sc._jvm.org.apache.hadoop.fs.Path(base_path_str)
fs = path_obj.getFileSystem(sc._jsc.hadoopConfiguration())

# Retrieve all item statuses inside the structured directory
file_statuses = fs.listStatus(path_obj)

valid_delta_paths = []
for status in file_statuses:
    # Filter for directories only (Delta tables are stored as directories)
    if status.isDirectory():
        full_path = status.getPath().toString()
        folder_name = full_path.split("/")[-1] or full_path.split("/")[-2]
        
        # CRITICAL FILTER: Skip 'raw', 'file_catalog' and hidden/metadata folders
        if folder_name in ["raw", "file_catalog"] or folder_name.startswith("."):
            continue
            
        valid_delta_paths.append(full_path)

In [ ]:
import re
from pyspark.sql import DataFrame
from pyspark.sql.types import StringType, IntegerType, LongType, FloatType, DoubleType, BooleanType, DateType, TimestampType, ShortType, ByteType, ArrayType

def extract_clean_dataframe_and_ddl(spark_session: SparkSession, s3_path: str, database_name: str = "bi_analytics") -> tuple[DataFrame, str]:

    folder_name = s3_path.split("/")[-1] or s3_path.split("/")[-2]
    clean_table_name = folder_name.replace("_delta", "")
    clean_table_name = re.sub(r'_\d+$', '', clean_table_name)
    clean_table_name = clean_table_name.replace("-", "_").lower()
    
    sc = spark_session.sparkContext
    delta_log_path_str = s3_path.rstrip("/") + "/_delta_log"
    delta_log_path_obj = sc._jvm.org.apache.hadoop.fs.Path(delta_log_path_str)
    fs = delta_log_path_obj.getFileSystem(sc._jsc.hadoopConfiguration())
    
    if fs.exists(delta_log_path_obj):
        df = spark_session.read.format("delta").load(s3_path)
    else:
        df = spark_session.read.parquet(s3_path)
        

    for col_name in df.columns:
        cleaned_col_name = col_name.replace("ï»¿", "").replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")
        if cleaned_col_name != col_name:
            df = df.withColumnRenamed(col_name, cleaned_col_name)
            
    sorting_keys = []
    if "global_warming" in clean_table_name:
        sorting_keys = ["Country", "Year"]
    elif "temperature_change" in clean_table_name:
        sorting_keys = ["Area", "Months"]
    elif "emission" in clean_table_name:
        sorting_keys = ["Make", "Model"]
    elif "tweet" in clean_table_name:
        sorting_keys = ["tweet_id"]

    final_sorting_keys = [k for k in sorting_keys if k in df.columns]
    if not final_sorting_keys:
        final_sorting_keys = [k for k in ["id"] if k in df.columns]
    
    order_by_clause = ", ".join([f"`{k}`" for k in final_sorting_keys]) if final_sorting_keys else f"`{df.columns[0]}`"
    
    type_mapping = {
        StringType: "String", IntegerType: "Int32", LongType: "Int64",
        FloatType: "Float32", DoubleType: "Float64", BooleanType: "UInt8",
        DateType: "Date", TimestampType: "DateTime", ShortType: "Int16", ByteType: "Int8"
    }
    
    ch_columns = []
    for field in df.limit(0).schema.fields:
        if isinstance(field.dataType, ArrayType):
            element_type_class = type(field.dataType.elementType)
            ch_type = f"Array({type_mapping.get(element_type_class, 'String')})"
        else:
            ch_type = type_mapping.get(type(field.dataType), "String")
            
        if field.name in final_sorting_keys:
            ch_columns.append(f"    `{field.name}` {ch_type}")
        else:
            ch_columns.append(f"    `{field.name}` Nullable({ch_type})")
        
    columns_sql = ",\n".join(ch_columns)
    
    native_ddl = f"""CREATE TABLE IF NOT EXISTS {database_name}.{clean_table_name} (
{columns_sql}
) ENGINE = MergeTree()
ORDER BY ({order_by_clause})"""

    return df, native_ddl

In [ ]:
import clickhouse_connect

def load_dataframe_to_clickhouse(spark_session: SparkSession, df: DataFrame, ddl_sql: str, target_table_name: str, database_name: str = "bi_analytics"):


    client = clickhouse_connect.get_client(
        host='clickhouse',
        port=8123,
        username='analytics',
        password='analytics_secret',
        database=database_name
    )
    
    client.command(ddl_sql)
    
    full_table_path = f"{database_name}.{target_table_name}"

    data_rows = df.collect()
    column_names = df.columns
    
    if data_rows:
        insert_data = [tuple(row) for row in data_rows]
        
        client.insert(
            table=target_table_name,
            data=insert_data,
            column_names=column_names
        )

    else:
        print("empty data")
        
    client.close()

In [ ]:
for s3_delta_path in valid_delta_paths:
    folder_name = s3_delta_path.split("/")[-1] or s3_delta_path.split("/")[-2]
    tbl_name = re.sub(r'_\d+$', '', folder_name.replace("_delta", "")).replace("-", "_").lower()


    try:
        cleaned_df, clickhouse_ddl = extract_clean_dataframe_and_ddl(spark, s3_delta_path)
        

        load_dataframe_to_clickhouse(spark, cleaned_df, clickhouse_ddl, target_table_name=tbl_name)
        
    except Exception as e:
        continue

**Applying Transformations**

Next, we will use Spark to read the files from MinIO and apply some generic transformations on Parquet files. After that, we import them into MongoDB. The following functions can help us do some generic transformations on the data.

In [ ]:
# ----------------------------
# CLEAN COLUMN NAMES
# ----------------------------
def clean_column(name):
    name = name.encode("ascii", "ignore").decode()
    name = name.lower().strip()
    name = re.sub(r"[^\w]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")

# ----------------------------
# CLEAN TABLE NAMES
# ----------------------------
def clean_table_name(name):
    name = name.replace("-", "_")
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)
    return name.lower()

# ----------------------------
# PERIOD NORMALIZATION (KEEP ONLY THIS VERSION)
# ----------------------------
def normalize_period_column(df, col):

    if col not in df.columns:
        return df

    x = F.lower(F.col(col))

    x = F.regexp_replace(x, r"[^\w\s\-]", "-")
    x = F.regexp_replace(x, r"-+", "-")
    x = F.trim(x)

    return df.withColumn(
        col,
        F.when(x.isNull(), None)
         .otherwise(x)
    )

In [ ]:
# LIST ALL "FOLDERS"
response = s3.list_objects_v2(
    Bucket="landing-zone",
    Prefix="persistent-landing/structured/",
    Delimiter="/"
)

folders = []
for prefix in response.get("CommonPrefixes", []):
    path = prefix["Prefix"]

    # exclude unwanted folders
    if "raw" in path or "file_catalog" in path:
        continue

    folders.append(path)

datasets = {}

# PROCESS EACH DATASET
for folder in folders:

    raw_table_name = folder.split('/', 3)[2]
    table_name = clean_table_name(raw_table_name)

    print(f"\nProcessing: {table_name}")

    # 1. Read parquet
    df = spark.read.parquet(f"s3a://landing-zone/{folder}")

    # 2. Drop duplicates
    df = df.dropDuplicates()

    # 3. Standardize column names
    df = df.toDF(*[clean_column(c) for c in df.columns])

    # 4. Fix period/month column
    for c in ["months", "month", "period"]:
        if c in df.columns:
            df = normalize_period_column(df, c)

    # 5. Schema cleanup
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, F.lower(F.trim(F.col(c))))

    # 6. Conversion point
    pdf = df.toPandas()
    pdf = pdf.where(pd.notna(pdf), None)
    records = pdf.to_dict("records")
    collection = db[table_name]

    # 7. Clean insert
    if records:
        collection.delete_many({})

        batch_size = 5000
        for i in range(0, len(records), batch_size):
            collection.insert_many(records[i:i + batch_size], ordered=False)

    print(f"Loaded {table_name}: {len(records)} rows")

In [ ]:
# Check inserted collections
db = client["trusted_zone_structured"]
print(db.list_collection_names())

**Showing Transformed Data**

In [ ]:
name = next(c for c in db.list_collection_names()
            if c.startswith("co2_emission_by_vehicles"))
df = pd.DataFrame(list(db[name].find({}, {"_id": 0})))
df.head(5)

In [ ]:
name = next(c for c in db.list_collection_names()
            if c.startswith("global_warming"))
df = pd.DataFrame(list(db[name].find({}, {"_id": 0})))
df.head(5)

In [ ]:
name = next(c for c in db.list_collection_names()
            if c.startswith("natural_disaster_tweets"))
df = pd.DataFrame(list(db[name].find({}, {"_id": 0})))
df.head(5)

In [ ]:
name = next(c for c in db.list_collection_names()
            if c.startswith("temperature_change"))
df = pd.DataFrame(list(db[name].find({}, {"_id": 0})))
df.head(5)

We have nested lists in the columns `hashtags` and `emojis` of the collection `natural_disaster_tweets`, but due to the high number of labels, we decided not to flatten them into new columns.

In [ ]:
# Print the number of unique elements in a nested column
def print_num_unique_labels(df, column):

    def parse_cell(x):
        if pd.isna(x):
            return []

        if isinstance(x, list):
            return x

        if isinstance(x, str):
            try:
                return ast.literal_eval(x)
            except:
                cleaned = x.replace("[", "").replace("]", "").replace("'", "")
                return [i.strip() for i in cleaned.split(",") if i.strip()]

        return []

    all_tags = df[column].apply(parse_cell).explode()
    all_tags = all_tags.dropna().astype(str).str.strip().str.lower()

    print(f"Number of unique labels in {column}: {all_tags.nunique()}")

# Get collection natural_disaster_tweets
name = next(c for c in db.list_collection_names()
            if c.startswith("natural_disaster_tweets"))
df = pd.DataFrame(list(db[name].find({}, {"_id": 0})))

print_num_unique_labels(df, "hashtags")
print_num_unique_labels(df, "emojis")